In [ ]:
rm(list=ls())
library(CellChat)
library(patchwork)
library(Seurat)
options(stringsAsFactors = FALSE)
options(future.globals.maxSize = 20 * 1024^3) # why did I have to add this? 

In [ ]:
sessionInfo()

In [ ]:
seurat_path = '/home/EOCRC_atlas/data/all_samples_raw_withTier2Annotation_09-19-25.rds'
save_path = '/home/EOCRC_atlas/results/EOCRC_CellChat_nboot1000/'
if (!dir.exists(save_path)) {dir.create(save_path, recursive = TRUE)}

# CellChat "UnderFifty" samples

In [ ]:
# Load Seurat object
crc <- readRDS(seurat_path)

In [ ]:
# subset to just the cell types we want 
keep = c('Adipocytes', 'B cell', 'CD4 T cells', 'CD8 T cells',
       'CEACAM1 colonocyte-like', 'Cycing endothelium', 'Cycling Myeloid',
       'Cycling Stromal', 'Cycling T cells', 'Cycling plasma cell', 'DC',
       'Enteroendocrine-like', 'Fibroblast', 'Fibroblast-BMP5-SOX6',
       'Fibroblast-C3', 'Fibroblast-Infl', 'Fibroblast-KCNN3',
       'Fibroblast-MMP2-THY1', 'Germinal center / Cycling B cell',
       'Glial cells', 'HSP-hi - B cell', 'HSP-hi Myeloid',
       'HSP-hi Stromal', 'HSP-hi T cells', 'HSP-hi glial', 'ILCs',
       'LGR5 stem cell-like', 'Lymphatic endothelium',
       'MT-Ribo-hi Myeloid', 'MT-Ribo-hi Stromal', 'MT-Ribo-hi T cells',
       'MT-Ribo-hi endothelium', 'MT-Ribo-hi epithelial',
       'MUC2 goblet-like', 'Macrophage-Monocyte', 'Mast',
       'Myofibroblast-SMC', 'NK-Cytotoxic T cells', 'Neuronal cells',
       'Neutrophil', 'Patient-specific', 'Pericytes', 'Plasma cell',
       'Regulatory T cells', 'T helper cells', 'Vascular endothelium')
crc = subset(crc, subset = Annotation_Tier2%in%keep)

In [ ]:
unique(crc@meta.data['Annotation_Tier2'][crc@meta.data['Annotation_Tier1']=="Epithelial"])

In [ ]:
# create Annotation Tier 1.5 
crc@meta.data$Annotation_1.5 = crc@meta.data$Annotation_Tier1
levels(crc@meta.data$Annotation_1.5) <- c(levels(crc@meta.data$Annotation_1.5), c('CEACAM1 colonocyte-like','MT-Ribo-hi epithelial','MUC2 goblet-like','LGR5 stem cell-like','Enteroendocrine-like','Patient-specific'))
crc@meta.data$Annotation_1.5[crc@meta.data$Annotation_1.5 == "Epithelial"] = crc@meta.data$Annotation_Tier2[crc@meta.data$Annotation_1.5 == "Epithelial"]
crc@meta.data$Annotation_1.5 <- factor(crc@meta.data$Annotation_1.5)
unique(crc@meta.data$Annotation_1.5)

In [ ]:
# subset CRC to just the MSS left samples
crc = subset(crc, subset = MSI_v2 == "MSS: STABLE")
crc = subset(crc, subset = Sidedness == "Left")

In [ ]:
# Confirm that we only have left MSS samples 
unique(crc@meta.data$Sidedness)
unique(crc@meta.data$MSI_v2)

In [ ]:
# Subset to just under 50 
crc = subset(crc, subset = Cohort == "UnderFifty")

In [ ]:
# Normalize data 
crc <- NormalizeData(crc, normalization.method = "LogNormalize", scale.factor = 10000)

In [ ]:
# check that we have the young patients only 
print(length(unique(crc@meta.data$FRID))) 
unique(crc@meta.data$Cohort)

In [ ]:
# Create counts matrix as matrix 
counts <- as.matrix(GetAssayData(crc, assay = "RNA", slot = "data"))

In [ ]:
# Get annotation information 
meta <- crc@meta.data
meta$labels <- meta[["Annotation_1.5"]] 
meta$labels <- droplevels(meta$labels)
print(unique(meta$labels)) # check the cell labels

# cellchat gives a warning about creating a single sampleid if it doesn't find a "samples" column
meta$samples = meta$FRID
head(meta)

In [ ]:
cellchat <- createCellChat(object = counts, meta = meta, group.by = "labels")

In [ ]:
CellChatDB <- CellChatDB.human # use CellChatDB.mouse if running on mouse data
CellChatDB.use <- subsetDB(CellChatDB)

In [ ]:
unique(CellChatDB.use$interaction$annotation)

In [ ]:
cellchat@DB <- CellChatDB.use

In [ ]:
ptm = Sys.time()
# subset the expression data of signaling genes for saving computation cost
cellchat <- subsetData(cellchat) # This step is necessary even if using the whole database
cellchat <- identifyOverExpressedGenes(cellchat)
cellchat <- identifyOverExpressedInteractions(cellchat)

In [ ]:
execution.time = Sys.time() - ptm
print(as.numeric(execution.time, units = "secs"))

In [ ]:
ptm = Sys.time()
options(future.globals.maxSize = 8000 * 1024^2) 
cellchat <- computeCommunProb(cellchat, type = "triMean", nboot = 1000)

In [ ]:
cellchat <- filterCommunication(cellchat, min.cells = 10) # why was this step ommitted in Bin's code? 

In [ ]:
cellchat <- computeCommunProbPathway(cellchat) # why was this step ommitted in Bin's code? 

In [ ]:
cellchat <- aggregateNet(cellchat)
execution.time = Sys.time() - ptm
print(as.numeric(execution.time, units = "secs"))

In [ ]:
rm(counts, crc)
gc()

In [ ]:
save(cellchat, file = paste0(save_path, "/cellchat_Tier1.5_MSS-LEFT_noMixed_under50_09-15-2025.RData"))

# CellChat on FiftyPlus

In [ ]:
# Load Seurat object to avoid another conversion from AnnData 
crc <- readRDS(seurat_path)

In [ ]:
# subset to just the cell types we want 
keep = c('Adipocytes', 'B cell', 'CD4 T cells', 'CD8 T cells',
       'CEACAM1 colonocyte-like', 'Cycing endothelium', 'Cycling Myeloid',
       'Cycling Stromal', 'Cycling T cells', 'Cycling plasma cell', 'DC',
       'Enteroendocrine-like', 'Fibroblast', 'Fibroblast-BMP5-SOX6',
       'Fibroblast-C3', 'Fibroblast-Infl', 'Fibroblast-KCNN3',
       'Fibroblast-MMP2-THY1', 'Germinal center / Cycling B cell',
       'Glial cells', 'HSP-hi - B cell', 'HSP-hi Myeloid',
       'HSP-hi Stromal', 'HSP-hi T cells', 'HSP-hi glial', 'ILCs',
       'LGR5 stem cell-like', 'Lymphatic endothelium',
       'MT-Ribo-hi Myeloid', 'MT-Ribo-hi Stromal', 'MT-Ribo-hi T cells',
       'MT-Ribo-hi endothelium', 'MT-Ribo-hi epithelial',
       'MUC2 goblet-like', 'Macrophage-Monocyte', 'Mast',
       'Myofibroblast-SMC', 'NK-Cytotoxic T cells', 'Neuronal cells',
       'Neutrophil', 'Patient-specific', 'Pericytes', 'Plasma cell',
       'Regulatory T cells', 'T helper cells', 'Vascular endothelium')
crc = subset(crc, subset = Annotation_Tier2%in%keep)

In [ ]:
# create Annotation Tier 1.5 
crc@meta.data$Annotation_1.5 = crc@meta.data$Annotation_Tier1
levels(crc@meta.data$Annotation_1.5) <- c(levels(crc@meta.data$Annotation_1.5), c('CEACAM1 colonocyte-like','MT-Ribo-hi epithelial','MUC2 goblet-like','LGR5 stem cell-like','Enteroendocrine-like','Patient-specific'))
crc@meta.data$Annotation_1.5[crc@meta.data$Annotation_1.5 == "Epithelial"] = crc@meta.data$Annotation_Tier2[crc@meta.data$Annotation_1.5 == "Epithelial"]
crc@meta.data$Annotation_1.5 <- factor(crc@meta.data$Annotation_1.5)
unique(crc@meta.data$Annotation_1.5)

In [ ]:
# subset CRC to just the left samples and then compare under / over fifty on just the left side 
crc = subset(crc, subset = MSI_v2 == "MSS: STABLE")
crc = subset(crc, subset = Sidedness == "Left")

In [ ]:
# Check that they are all MSS left
unique(crc@meta.data$MSI_v2)
unique(crc@meta.data$Sidedness)

In [ ]:
# Subset to just under 50 
crc = subset(crc, subset = Cohort == "FiftyPlus")

In [ ]:
# Normalize data 
crc <- NormalizeData(crc, normalization.method = "LogNormalize", scale.factor = 10000)

In [ ]:
# check that we have the young patients only 
print(length(unique(crc@meta.data$FRID))) 
unique(crc@meta.data$Cohort)

In [ ]:
# Create counts matrix as matrix 
counts <- as.matrix(GetAssayData(crc, assay = "RNA", slot = "data"))

In [ ]:
# Get Tier1 annotation information 
meta <- crc@meta.data
meta$labels <- meta[["Annotation_1.5"]] 
meta$labels <- droplevels(meta$labels)
print(unique(meta$labels)) # check the cell labels

# cellchat gives a warning about creating a single sampleid if it doesn't find a "samples" column
meta$samples = meta$FRID
head(meta)

In [ ]:
cellchat <- createCellChat(object = counts, meta = meta, group.by = "labels")

In [ ]:
CellChatDB <- CellChatDB.human # use CellChatDB.mouse if running on mouse data
CellChatDB.use <- subsetDB(CellChatDB)

In [ ]:
cellchat@DB <- CellChatDB.use

In [ ]:
ptm = Sys.time()
# subset the expression data of signaling genes for saving computation cost
cellchat <- subsetData(cellchat) # This step is necessary even if using the whole database
cellchat <- identifyOverExpressedGenes(cellchat)
cellchat <- identifyOverExpressedInteractions(cellchat)

In [ ]:
execution.time = Sys.time() - ptm
print(as.numeric(execution.time, units = "secs"))

In [ ]:
ptm = Sys.time()
options(future.globals.maxSize = 8000 * 1024^2)
cellchat <- computeCommunProb(cellchat, type = "triMean", nboot = 1000)

In [ ]:
cellchat <- filterCommunication(cellchat, min.cells = 10) # why was this step ommitted in Bin's code? 

In [ ]:
cellchat <- computeCommunProbPathway(cellchat) # why was this step ommitted in Bin's code? 

In [ ]:
cellchat <- aggregateNet(cellchat)
execution.time = Sys.time() - ptm
print(as.numeric(execution.time, units = "secs"))

In [ ]:
rm(crc, counts)
gc()

In [ ]:
save(cellchat, file = paste0(save_path, "/cellchat_Tier1.5_MSS-LEFT_noMixed_50Plus_09-15-2025.RData"))

# Merge the two cellchat objects and do a comparison analysis 

In [ ]:
seurat_path = '/home/EOCRC_atlas/data/all_samples_raw_withTier2Annotation_09-19-25.rds'
save_path = '/home/EOCRC_atlas/results/EOCRC_CellChat_nboot1000/'

load(paste0(save_path, "/cellchat_Tier1.5_MSS-LEFT_noMixed_under50_09-15-2025.RData"))
cellchat.underFifty = cellchat 
load(paste0(save_path, "/cellchat_Tier1.5_MSS-LEFT_noMixed_50Plus_09-15-2025.RData"))
cellchat.fiftyPlus = cellchat
rm(list = ls()[!ls() %in% c("cellchat.underFifty", "cellchat.fiftyPlus", "save_path")])

object.list <- list(fiftyPlus = cellchat.fiftyPlus, underFifty = cellchat.underFifty)
cellchat <- mergeCellChat(object.list, add.names = names(object.list))

In [ ]:
# update save path so as not to overwrite old results - making new connector plots
save_path = '/home/EOCRC_atlas/results/EOCRC_CellChat_nboot1000/'

In [ ]:
ptm = Sys.time()
gg1 <- compareInteractions(cellchat, show.legend = F, group = c(1,2))
gg2 <- compareInteractions(cellchat, show.legend = F, group = c(1,2), measure = "weight")
gg1 = gg1 + scale_fill_manual(values = c('#E69F00', '#56B4E9'))  
gg1
ggsave(paste0(save_path, "figures/cellchat_Tier1.5_CompareInteractions_MSS_LEFT_noMixed.pdf"), plot = gg1, width = 3, height = 6, units = "in")

In [ ]:
library(grid)
grid.newpage()
gg1 <- netVisual_heatmap(cellchat)
gg1
dev.copy(pdf, paste0(save_path, "figures/cellchat_Tier1.5_differentialNumberHM_MSS_LEFT_noMixed.pdf"), width = 6, height = 6)
dev.off()

In [ ]:
gg1 <- rankNet(cellchat, mode = "comparison", measure = "weight", sources.use = NULL, targets.use = NULL, stacked = T, do.stat = TRUE)
gg2 <- rankNet(cellchat, mode = "comparison", measure = "weight", sources.use = NULL, targets.use = NULL, stacked = F, do.stat = TRUE)

gg1 = gg1 + scale_fill_manual(values = c('#56B4E9', '#E69F00'))  
gg1
ggsave(paste0(save_path, "figures/cellchat_Tier1.5_RankNet_MSS_LEFT_noMixed.pdf"), plot = gg1, width = 5, height = 8, units = "in")

gg2 = gg2 + scale_fill_manual(values = c('#56B4E9', '#E69F00'))  
gg2
ggsave(paste0(save_path, "figures/cellchat_Tier1.5_RankNet2_MSS_LEFT_noMixed.pdf"), plot = gg2, width = 5, height = 8, units = "in")

In [ ]:
path = c('JAM', 'VISFATIN', 'PECAM2', 'COLLAGEN', 'LAMININ', 'CD45', 'CEACAM')
for (p in path){
            pathways.show <- p
            weight.max <- getMaxWeight(object.list, slot.name = c("netP"), attribute = pathways.show) # control the edge weights across different datasets
            par(mfrow = c(1,2), xpd=TRUE)
            for (i in 1:length(object.list)) {
              netVisual_aggregate(object.list[[i]], signaling = pathways.show, layout = "circle", edge.weight.max = weight.max[1], edge.width.max = 10, signaling.name = names(object.list)[i])
              title(main = pathways.show, cex.main = 1.5)
            }
            dev.copy(pdf, paste0(save_path, "figures/cellchat_Tier1.5_connectorPlot_", p, "_MSS_LEFT_noMixed.pdf"), width = 6, height = 6)
            dev.off()
    }

In [ ]:
path = c('CDH5', 'TENASCIN', 'FN1', 'PARs', 'SEMA3', 'CypA', 'PERIOSTIN', 'NRXN', 'SEMA6', 'ANNEXIN', 'CADM', 'APP', 'GDF', 'EGF', 
         'ADGRG', 'DESMOSOME', 'OCLN', 'CD99', 'CLDN')
for (p in path){
            pathways.show <- p
            weight.max <- getMaxWeight(object.list, slot.name = c("netP"), attribute = pathways.show) # control the edge weights across different datasets
            par(mfrow = c(1,2), xpd=TRUE)
            for (i in 1:length(object.list)) {
              netVisual_aggregate(object.list[[i]], signaling = pathways.show, layout = "circle", edge.weight.max = weight.max[1], edge.width.max = 10, signaling.name = names(object.list)[i])
              title(main = pathways.show, cex.main = 1.5)
            }
            dev.copy(pdf, paste0(save_path, "figures/cellchat_Tier1.5_connectorPlot_", p, "_MSS_LEFT_noMixed.pdf"), width = 6, height = 6)
            dev.off()
    }

In [ ]:
path = c('NECTIN', 'GRN', 'MIF', 'CD96', 'MHC-II', 'BMP', 'ANGPTL', 'SPP1', 'ACTIVIN')
    for (i in path){
        pathways.show <- i
        # Circle plot
        par(mfrow=c(1,1))
        netVisual_aggregate(object.list[[2]], signaling = pathways.show, layout = "circle")
        title(main = pathways.show, cex.main = 1.5)
        dev.copy(pdf, paste0(save_path, "figures/cellchat_Tier1.5_connectorPlot_", i, "_MSS_LEFT_UnderFifty_noMixed.pdf"), width = 6, height = 6)
        dev.off()
    }

In [ ]:
# extract info 
df.net <- subsetCommunication(object.list[[1]])
head(df.net)
write.csv(df.net, paste0(save_path, "figures/LR_pair_details_fiftyPlus.csv"))

df.net <- subsetCommunication(object.list[[2]])
head(df.net)
write.csv(df.net, paste0(save_path, "figures/LR_pair_details_underFifty.csv"))